# Baseline modele: LSTM i Transformer

Notebook definiuje pipeline do porównania baseline modeli dla rozpoznawania krótkich komend głosowych. Nie uruchamia eksperymentów automatycznie; konfiguracje, modele, dataset, trening i zapis wyników znajdują się w modułach `scripts`.

In [1]:
import pandas as pd

from scripts import (
    DataFixedParams,
    DataGridParams,
    Experiment,
    FeatureFixedParams,
    FitFixedParams,
    FitGridParams,
    LABEL_ORDER,
    ModelGridParams,
    experiment_grid_dataframe,
    prepare_experiment_datasets,
    train_experiment,
)

LABEL_ORDER

('yes',
 'no',
 'up',
 'down',
 'left',
 'right',
 'on',
 'off',
 'stop',
 'go',
 'unknown',
 'silence')

## Konfiguracja eksperymentu


In [2]:
# Wszystkie parametry eksperymentu są zebrane w tej komórce.
# Konfiguracja poniżej jest małym, ale użytecznym testem poprawności treningu.

feature_fixed = FeatureFixedParams(
    n_mels=64,
    n_fft=512,
    hop_length=160,
    normalize=True,
)

# data_grid = DataGridParams(
#     train_fraction=0.1,
#     validation_fraction=0.1,
#     test_fraction=0.1,
#     unknown_fraction=0.01,
#     silence_samples=50,
#     sampling_strategy="natural",
#     seed=42,
# )

data_grid = DataGridParams(
    train_fraction=0.6,
    validation_fraction=1,
    test_fraction=1,
    unknown_fraction=0.06,
    silence_samples=2000,
    sampling_strategy="natural",
    seed=42,
)

model_grid = ModelGridParams(
    model_type=["lstm", "transformer"],
    # model_type=["transformer"],
    dropout=0.2,
    lstm_hidden_size=32,
    lstm_layers=2,
    lstm_bidirectional=True,
    transformer_d_model=32,
    transformer_heads=4,
    transformer_layers=2,
    transformer_ff_dim=64,
)

fit_fixed = FitFixedParams(
    device="cuda",
    use_tqdm=True,
    progress_backend="terminal",
    verbose=True,
    early_stopping=True,
    early_stopping_patience=5,
    early_stopping_min_delta=0.001,
)
fit_grid = FitGridParams(
    epochs=10,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
)

data_fixed = DataFixedParams(
    reuse_cached_dataset=True,
)

baseline_experiment = Experiment(
    name="small_functional_baseline_lstm_transformer",
    data_fixed=data_fixed,
    data_grid=data_grid,
    feature_fixed=feature_fixed,
    model_grid=model_grid,
    fit_fixed=fit_fixed,
    fit_grid=fit_grid,
)

experiment_grid_dataframe(baseline_experiment)


,experiment,data.train_fraction,data.validation_fraction,data.test_fraction,data.unknown_fraction,data.silence_samples,data.sampling_strategy,data.seed,model.model_type,model.dropout,...,model.lstm_layers,model.lstm_bidirectional,model.transformer_d_model,model.transformer_heads,model.transformer_layers,model.transformer_ff_dim,fit.epochs,fit.batch_size,fit.learning_rate,fit.weight_decay
0,small_functional_baseline_lstm_transformer,0.6,1,1,0.06,2000,natural,42,lstm,0.2,...,2,True,32,4,2,64,10,64,0.001,0.0001
1,small_functional_baseline_lstm_transformer,0.6,1,1,0.06,2000,natural,42,transformer,0.2,...,2,True,32,4,2,64,10,64,0.001,0.0001


## Uruchomienie eksperymentu

Najpierw przygotuj dane i sprawdź obiekt `prepared_data`. Dopiero kolejna komórka uruchamia trening oraz końcowy test.


In [3]:
prepared_data = prepare_experiment_datasets(baseline_experiment)


Building dataset


Scanning archive: 100%|██████████| 64732/64732 [00:00<00:00, 102522.10it/s]


  -> samples | train=13247 | validation=3041 | test=3035
     class        train  validation  test
     down         1106         264   253
     go           1117         260   251
     left         1104         247   267
     no           1112         270   252
     off          1104         256   262
     on           1119         257   246
     right        1112         256   259
     silence       948         210   211
     stop         1131         246   249
     unknown      1172         254   257
     up           1106         260   272
     yes          1116         261   256


In [4]:
results = train_experiment(baseline_experiment, prepared_data)
results

Starting experiment: small_functional_baseline_lstm_transformer

Configuration run 1/2:
DATA (variable):
  - train_fraction: 0.6
  - validation_fraction: 1
  - test_fraction: 1
  - unknown_fraction: 0.06
  - silence_samples: 2000
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: lstm
  - dropout: 0.2
  - lstm_hidden_size: 32
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 32
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 64
FIT (variable):
  - epochs: 10
  - batch_size: 64
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cuda


Training: 100%|██████████| 10/10 [02:51<00:00, 17.15s/it, loss=0.4247, lr=0.001, val_acc=0.8451, val_loss=0.4826]


Training finished in 174.54 seconds



Configuration run 2/2:
DATA (variable):
  - train_fraction: 0.6
  - validation_fraction: 1
  - test_fraction: 1
  - unknown_fraction: 0.06
  - silence_samples: 2000
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: transformer
  - dropout: 0.2
  - lstm_hidden_size: 32
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 32
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 64
FIT (variable):
  - epochs: 10
  - batch_size: 64
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cuda


Training: 100%|██████████| 10/10 [03:20<00:00, 20.07s/it, loss=0.3958, lr=0.001, val_acc=0.8524, val_loss=0.4910]


Training finished in 203.45 seconds



Experiment finished | total runs = 2


,run,best_epoch,epochs_trained,stopped_early,train_loss,train_accuracy,validation_loss,validation_accuracy,test_loss,test_accuracy
0,lstm_train0_6_val1_test1_lr0_001_seed42,10,10,False,0.424664,0.868348,0.482560,0.845117,0.511244,0.843163
1,trfm_train0_6_val1_test1_lr0_001_seed42,9,10,False,0.422545,0.863441,0.477566,0.861559,0.482804,0.860297
